# fnb — Kaggle T4 Runner (skeleton)

Thin wrapper over `scripts/` for running the **Fake News Benchmark** on Kaggle (single NVIDIA T4).

**Loop:** author in Cursor → push to GitHub → run here → paste outputs back. See `README.md` §2 and `KAGGLE_WORKFLOW.md`.

**Before running:** in the right panel set **Accelerator** (GPU T4 x1 for training; None/CPU for tests + data pipeline), turn **Internet On**, and **Add Data** to attach DS1–DS5.

Run cells top-to-bottom (**Run All**), or run the section you need.

## 1. Clone + install (run at the start of every session)

Pulls the latest code from GitHub and installs the package editable. For a **private** repo, use a fine-grained read-only token in the URL.

In [ ]:
# --- EDIT THIS: your GitHub repo ---
REPO_URL = "https://github.com/<your-username>/fake-news-benchmark.git"
REPO_DIR = "fake-news-benchmark"
# Private repo? e.g. REPO_URL = "https://<TOKEN>@github.com/<your-username>/fake-news-benchmark.git"

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR
!pip install -e . -q
!python -c "import fnb; print('fnb', fnb.__version__)"

## 2. Attach + inspect input datasets

Datasets are attached via the right panel (**Add Data**); they mount **read-only** under `/kaggle/input/<slug>/`. This cell just lists what is attached so you can confirm the expected DS1–DS5 (and any re-attached output datasets) are present.

Outputs go to `/kaggle/working/` (read-write, **wiped at session end**).

In [ ]:
import os

INPUT_DIR = "/kaggle/input"    # read-only attached datasets
WORKING_DIR = "/kaggle/working"  # writable outputs (ephemeral)

print("Attached input datasets:")
if os.path.isdir(INPUT_DIR):
    for name in sorted(os.listdir(INPUT_DIR)):
        print("  -", os.path.join(INPUT_DIR, name))
else:
    print("  (none — not running on Kaggle, or no datasets attached)")

os.makedirs(WORKING_DIR, exist_ok=True)
print("\nOutputs will be written under:", WORKING_DIR)

## 3. Dispatch a stage / experiment via `scripts/`

The notebook holds **no research logic** — it only calls the thin CLIs in `scripts/`. Uncomment the command for what you want to run. Use **CPU** for tests + data pipeline, **GPU T4** for training experiments. See the M1→M8 run order in `README.md` §3.

In [ ]:
# --- Sanity: unit tests (CPU) ---
!pytest -q

# --- Data pipeline stage (CPU): EXP-P0 -> EXP-P5 ---
# !python scripts/run_data_pipeline.py --stage all

# --- Training experiment (GPU T4): dispatch any EXP-* by id ---
# !python scripts/run_experiment.py --exp EXP-1 --seed 13 --output-dir /kaggle/working

# --- Paper tables + figures (CPU) ---
# !python scripts/build_paper_outputs.py --results-dir results --out-dir /kaggle/working/paper

## 4. Save `/kaggle/working` outputs as a versioned dataset

`/kaggle/working/` is **wiped when the session ends**, so heavy outputs (processed parquet, checkpoints, QLoRA adapters) must be persisted as a **Kaggle output dataset** to reuse next session.

1. Right panel → **Output** → **Create Dataset** (or **Save Version**, which snapshots the output).
2. Name it, e.g. `fnb-outputs-milestone5`.
3. Next session → **Add Data** → attach it; it appears under `/kaggle/input/fnb-outputs-milestone5/`.

Small text artifacts (result CSVs, `SNAPSHOT_HASHES.txt`, split `*.idx`, logs) should be downloaded and committed back into the repo (see `KAGGLE_WORKFLOW.md` §4–§5). This cell lists what is currently in `/kaggle/working/` so you know what to save.

In [ ]:
import os

WORKING_DIR = "/kaggle/working"
print("Contents of", WORKING_DIR, "(to persist as an output dataset):")
for root, _dirs, files in os.walk(WORKING_DIR):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {size_mb:8.2f} MB  {path}")